# Lesson 000.22 -- Gluing Triangles Together: 2-Complexes, Orientation Cancellation, and Additivity of Flux

AuthorityEffect: None.

> **Claim-typing legend**
> - **[Pinned]**: reproduced from verified checked-in simplicial complex specifications.
> - **[Executed]**: verified by deterministic Python execution in this notebook.
> - **[Declared]**: pedagogical convention or beginner framing.
> - **[Open]**: questions left for future exploration.

## 1. Shared Edges and Orientation Cancellation

[Pinned] When two triangles share an edge, their boundaries cancel along the shared interface if orientations are matched:
- Triangle $F_1 = [0, 1, 2]$ has boundary $(0, 1) + (1, 2) + (2, 0)$.
- Triangle $F_2 = [1, 0, 3]$ has boundary $(1, 0) + (0, 3) + (3, 1)$.

The shared edge between 0 and 1 appears as $(0, 1)$ in $F_1$ and $(1, 0) = -(0, 1)$ in $F_2$.
In the sum $F_1 + F_2$:
$$(0, 1) + (1, 0) = 0.$$
Internal edges cancel, leaving only the exterior boundary cycle!

In [ ]:
# 1. Define two adjacent triangles sharing edge between 0 and 1
f1_edges = {(0, 1): 1, (1, 2): 1, (2, 0): 1}
f2_edges = {(1, 0): 1, (0, 3): 1, (3, 1): 1}

# 2. Combine into composite 2-chain boundary
combined_boundary = {}


def add_edge(d, u, v, coeff):
    if u > v:
        key = (v, u)
        sign = -coeff
    else:
        key = (u, v)
        sign = coeff
    d[key] = d.get(key, 0) + sign


for (u, v), coeff in f1_edges.items():
    add_edge(combined_boundary, u, v, coeff)
for (u, v), coeff in f2_edges.items():
    add_edge(combined_boundary, u, v, coeff)

# 3. Assert interior shared edge has cancelled identically to 0
assert combined_boundary[(0, 1)] == 0

# Remaining non-zero edges form the exterior 4-edge boundary
exterior = {e: c for e, c in combined_boundary.items() if c != 0}
assert len(exterior) == 4
assert set(exterior.keys()) == {(1, 2), (0, 2), (0, 3), (1, 3)}

print("Orientation cancellation on shared edge (0, 1) verified!")

## 2. Simplicial Stokes' Theorem: Additivity of Face Flux

[Pinned] Let $c \in C^1(K; \mathbb{Q})$ be an edge cochain.
The coboundary flux across the composite 2-chain $F_1 + F_2$ is additive:
$$d^1 c(F_1 + F_2) = d^1 c(F_1) + d^1 c(F_2) = \langle c, \partial_2(F_1 + F_2) 
angle.$$
The sum of interior fluxes equals the circulation along the exterior perimeter!

In [ ]:
from fractions import Fraction as Q

# Cochain assigning values to canonical edges (u < v)
c = {
    (0, 1): Q(7),  # Internal edge
    (1, 2): Q(3),
    (0, 2): Q(-2),  # (2, 0) in f1 has sign -1 on (0, 2)
    (0, 3): Q(4),
    (1, 3): Q(-1),  # (3, 1) in f2 has sign -1 on (1, 3)
}

# Flux across F1: c(0, 1) + c(1, 2) - c(0, 2)
flux_F1 = c[(0, 1)] * 1 + c[(1, 2)] * 1 + c[(0, 2)] * (-1)
# Flux across F2: -c(0, 1) + c(0, 3) - c(1, 3)
flux_F2 = c[(0, 1)] * (-1) + c[(0, 3)] * 1 + c[(1, 3)] * (-1)

total_flux = flux_F1 + flux_F2
boundary_circulation = sum(c[e] * coeff for e, coeff in combined_boundary.items())

assert flux_F1 == Q(12)
assert flux_F2 == Q(-2)
assert total_flux == boundary_circulation == Q(10)

print("Discrete Simplicial Stokes' Theorem verified with exact rationals!")

## 3. Diagnostic Probe (2026 Micro-Attention Check)

**Question:** Two adjacent triangles $F_1 = [0, 1, 2]$ and $F_2 = [1, 0, 3]$ share the edge between 0 and 1. What is the coefficient of edge $(0, 1)$ in the boundary $\partial_2(F_1 + F_2)$?

- **A:** 2, because both triangles touch that edge.
- **B:** 0, because $F_1$ traverses $0 \to 1$ and $F_2$ traverses $1 \to 0 = -(0, 1)$, canceling out in the sum.
- **C:** $-1$, because $F_2$ reverses it.

*(Correct answer: **B**)*


In [ ]:
def diagnostic_probe(choice: str) -> bool:
    return choice.strip().upper() == "B"


assert diagnostic_probe("B")
assert not diagnostic_probe("A")
assert not diagnostic_probe("C")

# Student Exercise: Form closed tetrahedron boundary (4 triangles)
# Show that every edge is shared with opposite orientations, so total boundary is zero
faces = [
    {(0, 1): 1, (1, 2): 1, (2, 0): 1},  # Face [0, 1, 2]
    {(0, 3): 1, (3, 1): 1, (1, 0): 1},  # Face [0, 3, 1]
    {(1, 3): 1, (3, 2): 1, (2, 1): 1},  # Face [1, 3, 2]
    {(2, 3): 1, (3, 0): 1, (0, 2): 1},  # Face [2, 3, 0]
]
closed_boundary = {}
for f in faces:
    for (u, v), coeff in f.items():
        add_edge(closed_boundary, u, v, coeff)

# Assert all 6 edges cancelled to 0!
assert all(coeff == 0 for coeff in closed_boundary.values())
assert len(closed_boundary) == 6

print("Lesson 000.22 diagnostic probe and closed 2-cycle exercise verified successfully!")